In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Learning values from samples: Monte Carlo and TD

## Where this session fits

> **The limitation we start from: dynamic programming needs the model of the environment (the state transition probabilities $\transpm$)**

- **This session's goal:** learn state values on Frozen Lake from episodes alone
- **Position: Part A (value-based methods), session 3**
    - values and the Bellman equation, then dynamic programming (a known model)
    - then **learning from samples (this session)**
    - then acting (control), then a network instead of a table
    - this session removes the model and keeps everything else
- **Series goal:** choose good actions by trial and error, from small tables to robots and language models

**By the end you can**

1. compute a Monte Carlo and a TD(0) update by hand
2. say what *bias* and *variance* of a target mean, and which of them Monte Carlo and TD each suffer from
    - both are defined in the interlude after Monte Carlo
3. say how far ahead a target looks (its depth) and how many successors it uses (its width), both defined in the backup-shape interlude
4. say why learning from samples needs many episodes


**Recall: the two exact forms of a value**

> **This session replaces the average over the model by one sample, in either form.**

- the trajectory form: $\statevalfun_\pi(\state) = \Exp{\pi}\left[ \rlret_\tt \mid \stateseq_\tt = \state \right]$, with $\rlret_\tt = \sum_{k \geq 0} \disc^k \, \rewseq_{\tt+k+1}$
- the one-step form (the Bellman equation), which needs the model $\transpm$:
$$
\statevalfun_\pi(\state) = \sum_{\act} \pi(\act \mid \state) \sum_{\state', \rew} \transpm(\state', \rew \mid \state, \act)\bigl(\rew + \disc\,\statevalfun_\pi(\state')\bigr)
$$
- Monte Carlo samples the first form, TD(0) samples the second
    - $\traj$ is one episode, and one trajectory gives one value of $\rlret_\tt$: a single sample
    - the next session meets this idea again for action values

# Learning from samples

**Why this section?** Last session's methods needed the model $\transpm$ (the state transition probabilities, with their rewards). Without it, the only source of information is experience.

> **Without a model, the only source of information is experience: playing episodes and recording what happened.**

- we learn about the dynamics of the Environment by *interacting* with it
    - playing out episodes (trajectories)
    - gathering rewards and successor states
- each *step* of an episode is a *sample* of the next reward and next state
    - drawn from an unknown transition distribution $\transpm(\state', \rew \mid \state, \act)$
    - an episode is a chain of such samples

**A state the episodes never reach is never updated**

- the episodes tell us nothing about it
- so the agent must *visit* the states it wants values for: this is **exploration**
- a model-based method needs none, because it never plays an episode
- this means multiple episodes are necessary
    - each state must be visited many times, and each outcome must show up often enough, in its right proportion
- contrast with model-based methods: all knowledge of the Environment is available without interacting

**Any state, or only the states of this episode?**

> **Model known: $\state$ and $\act$ stand for *any* state and action. From samples: $\stateseq_\tt$ and $\actseq_\tt$ are only the ones that *occurred* at step $\tt$ of this episode.**

<table>
<thead><tr><th nowrap><div align="left"></div></th><th nowrap><div align="left">model known (last session)</div></th><th nowrap><div align="left">from samples (this session)</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">state, action</div></td><td nowrap><div align="left">$\state$, $\act$: <i>any</i> of them; the update ranges over all</div></td><td nowrap><div align="left">$\stateseq_\tt$, $\actseq_\tt$: the ones the episode visited at step $\tt$</div></td></tr>
<tr><td nowrap><div align="left">next reward, next state</div></td><td nowrap><div align="left">$\rew$, $\state'$: <i>every</i> outcome, each weighted by $\transpm$</div></td><td nowrap><div align="left">$\rewseq_{\tt+1}$, $\stateseq_{\tt+1}$: the <i>one</i> outcome that occurred</div></td></tr>
<tr><td nowrap><div align="left">states an update touches</div></td><td nowrap><div align="left">every state, in every sweep</div></td><td nowrap><div align="left">only the states on the episode</div></td></tr>
<tr><td nowrap><div align="left">the letters</div></td><td nowrap><div align="left">lowercase: a particular value, not random</div></td><td nowrap><div align="left">capital: random, since another episode would give others</div></td></tr>
</tbody></table>

- $\rlret_\tt$ is the return from step $\tt$ of the episode, the trajectory $\traj$ (also random)
- watch for this shift of letters from here on: it marks the move from "all" to "what happened"

**We are not trying to estimate $\transpm$**
- what we want is the *expected return* of a state (its value), not the model
- we get it by averaging the returns we observe after the state
    - an outcome that happens often appears often in the average, so *how often it repeats* carries its probability
    - the probabilities are never recorded or estimated
    - they are built into the frequencies
- so sampling $(\rew, \state')$ pairs is a way to average, not a way to learn $\transpm$


### The one idea of this session

> **Replace an expectation by a sample.**

- a value is an *expectation*: an average over everything the environment could do, each outcome weighted by its probability
- with the model (last session) the average is a sum over all outcomes weighted by $\transpm$
- without it, replace the average by *one outcome*: the single one the environment *did* choose, instead of all the ones it *could* choose
- averaging many such outcomes recovers the expectation

# Monte Carlo (MC) method

**Why start with Monte Carlo?** It is the simplest way to learn from experience: wait for the episode to end and use the real return.

> **Monte Carlo: play the episode to the end, then update every state it visited.**

- play an episode with the policy until it reaches a terminal square
- for each visited state, find the **return from that state**: the rewards actually received after it, discounted
- the update must wait until the episode has ended
- if a state appears twice in an episode, we use the return from its *first* visit only
    - first-visit Monte Carlo: its target is exactly unbiased (*unbiased* is defined in the interlude after this slide)


**The Monte Carlo target and the update**

> **A target is the number that one update moves the current estimate toward: a single sample of what the value should be. For Monte Carlo the target is the return $\rlret_\tt$ from that state.**

$$
\statevalfun(\stateseq_\tt) \leftarrow \statevalfun(\stateseq_\tt) + \underbrace{\alpha \bigl( \rlret_\tt - \statevalfun(\stateseq_\tt) \bigr)}_{\text{increment}}
$$

- read it as *current value plus an increment*
    - the bracket is the **error**: target minus current estimate
    - the increment is $\alpha$ times the error, added to the current value
- $\alpha$ is a learning rate: the estimate moves part of the way toward the target (we say why part of the way below)
- the target is a real sample of the return: it contains no value estimate
- so Monte Carlo samples the **trajectory form**


A Monte Carlo update by hand, on (non-slippery) Frozen Lake
- non-slippery: every move goes where it is aimed
- squares are numbered row by row, $0$ (start) to $15$ (goal $\mathbf{G}$)
- one episode, played by some policy, follows the route $0 \to 4 \to 8 \to 9 \to 13 \to 14 \to \mathbf{G}$ (reward $+1$ on the last move)
- each visited state is updated toward *its own* return, discounted: $\disc = 0.9$
- all values start at $0$ and $\alpha = 0.5$, so the new value is $0.5 \, \rlret_\tt$

What do squares $14$ and $0$ become?


<table>
<thead><tr><th nowrap><div align="left">state</div></th><th nowrap><div align="center">$0$</div></th><th nowrap><div align="center">$4$</div></th><th nowrap><div align="center">$8$</div></th><th nowrap><div align="center">$9$</div></th><th nowrap><div align="center">$13$</div></th><th nowrap><div align="center">$14$</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">return $\rlret_\tt$</div></td><td nowrap><div align="center">$0.59$</div></td><td nowrap><div align="center">$0.66$</div></td><td nowrap><div align="center">$0.73$</div></td><td nowrap><div align="center">$0.81$</div></td><td nowrap><div align="center">$0.90$</div></td><td nowrap><div align="center">$1.00$</div></td></tr>
<tr><td nowrap><div align="left">updated value</div></td><td nowrap><div align="center">$0.30$</div></td><td nowrap><div align="center">$0.33$</div></td><td nowrap><div align="center">$0.36$</div></td><td nowrap><div align="center">$0.41$</div></td><td nowrap><div align="center">$0.45$</div></td><td nowrap><div align="center">$0.50$</div></td></tr>
</tbody></table>

- the return from square $0$ is $\disc^{5} = 0.9^5 \approx 0.59$: the reward arrives on the sixth move, and the first reward step is not discounted
- on a *slippery* lake the same square would see different returns in different episodes: the interlude after this slide looks at that


# Interlude: how good is a target? Bias and variance

**Why this interlude?** Recall: a *target* is the number one update moves the estimate toward (for Monte Carlo, the return). It is a random number: it depends on which episode happened.

> **Two questions say how good a random target is. We ask them of every method from here on.**
>
> 1. **Bias:** on average, does the target point at the true value?
> 2. **Variance:** how much does the target fluctuate?

- **Example:** the Monte Carlo target of the start square of the *slippery* lake
    - in one episode the agent reaches the goal quickly (a return up to $0.59$)
    - in another it falls into a hole (a return of $0$)
    - each episode gives a different number

**Question 1, bias: on average, does the target point at the true value?**

- **bias** = (the average of the target) $-$ (the true value)
- *unbiased* means the bias is zero
- **why it matters: bias pushes the estimate to the wrong place**
    - more episodes do not repair it: averaging many targets that all point to the wrong place gives a wrong average
    - (a bias can shrink only if the target itself improves, as TD's does below)

**Question 2, variance: how much does the target fluctuate around its own average?**

- **variance** = the average squared distance from the target's own average
- its square root is the **standard deviation** (std), the "spread" used in the pictures below
- **why it matters: variance makes every update jump**
    - the estimate follows the last few targets instead of their average
    - averaging many targets (or a small $\alpha$) reduces it, at the cost of more episodes
- **so the learning rate $\alpha$ has a job: move only part of the way**
    - a single target is a random sample
    - $\alpha = 1$ would jump all the way to it

**Monte Carlo, by these two questions**

- **bias:** none. The target is the actual return, and the value is *defined* as the average return
- **variance:** large. The return depends on every random step of the episode (where the agent slips, whether it falls into a hole)
    - a state $D$ steps from the end of the episode has a target built from $D$ random steps, and each one can change the outcome
    - so the farther the state is from the end, the more random quantities enter its target, and the more it fluctuates

> **The question this raises: can a target fluctuate less, even if we pay for it elsewhere?**


# Temporal Difference (TD) methods

**Why TD?** Monte Carlo must wait for the episode to end, and its target (a whole return) fluctuates a lot. Can we learn after every step, using what we already know?

> **Bootstrapping: building an estimate out of another estimate.** Here, a target built from the current estimate of the next state's value.

- Temporal Difference (TD) is a family of methods that learn values, characterized by bootstrapping
- the one we use first is **TD(0)** (one-step TD; "TD(0)" is just its name)
    - it is the sampled version of last session's policy evaluation: the Bellman *average* is replaced by *one sample*, so it samples the one-step form
    - it learns $\statevalfun(\state)$ for the policy that generates the episodes
    - the sampled version of value iteration, Q-learning, comes in the next session

After a step from $\stateseq_\tt$ (reward $\rewseq_{\tt+1}$, next state $\stateseq_{\tt+1}$) form the **target**
$$
\rewseq_{\tt+1} + \disc \, \statevalfun(\stateseq_{\tt+1})
$$
- the immediate reward actually received
- plus the discounted *current estimate* of the successor's value
- $\statevalfun$ is the current table of estimates
- it changes as the updates arrive
- **this is bootstrapping**: $\statevalfun(\stateseq_{\tt+1})$ is an *estimate* that stands in for the rest of the return, $\disc\,\rlret_{\tt+1}$
    - Monte Carlo waits for that rest to happen and uses the actual $\rlret_\tt$, which contains no estimate

In contrast to model-based methods
- where the update averages over all actions and successors, because the probabilities are known

the model-free target uses the *single* action and successor that actually occurred.


The **temporal difference error** is the target minus the current estimate
$$
\delta = \rewseq_{\tt+1} + \disc \, \statevalfun(\stateseq_{\tt+1}) - \statevalfun(\stateseq_{\tt})
$$

Temporal differencing moves the estimate *toward the target*
$$
\statevalfun(\stateseq_{\tt}) \leftarrow \statevalfun(\stateseq_{\tt}) + \alpha \, \delta
$$
- $\alpha$ is the learning rate, as for Monte Carlo
- each step updates only the state just left, using the *current* estimate of the next state
- the value of a terminal state is $0$ (nothing follows it)

This differs from the model-based sweeps of the last session
- they updated *every* state from the previous table
- TD updates only the states the episode actually visits


**MC versus TD(0) on a two-state episode**

<img src="images/a3_chain_setup.png" width=40%>

- the only reward, $+1$, is earned on the last transition
- discount $\disc = 1$ and learning rate $\alpha = 1$ (chosen only so that the arithmetic is visible)

**What are the values after episode $1$, under MC? under TD(0)?**


**After episode 1: Monte Carlo versus TD(0)**

<img src="images/a3_chain_answer.png" width=55%>

- **MC moves the reward along the whole path in one episode**
- **TD(0) moves it one step per episode, but does not wait for the episode to end**

**How fast does the reward reach the table? Dynamic programming and sampling**

<img src="images/a3_reward_spread.png" width="700">

- non-slippery lake, discount $0.9$; the only reward is $+1$ for entering the goal
- **dynamic programming:** the first sweep already makes one square positive
    - all $11$ non-terminal squares are positive after $6$ sweeps
- **sampling:** nothing changes until an episode happens to reach the goal, about $50$ episodes at the median
    - TD(0) then passes the value back one square per visit, and reaches all $11$ squares within $1000$ episodes
    - Monte Carlo updates a whole route at once, but only for squares on an episode that ends at the goal
    - about $7.6$ of $11$ squares after $1000$ episodes
- each sampled curve is the mean of $300$ runs


How much does each target fluctuate, and does it point at the true value?
- the Monte Carlo target for $\state_1$ depends on *both* steps of the episode
- the TD(0) target for $\state_1$ depends on *one* random step (the reward and next state just seen)
- the rest of the return is replaced by a table entry, $\statevalfun(\state_2)$, a fixed number at the time of the update
    - so, with $D$ steps to go, MC has $D$ random steps in its target and TD(0) has one: **low variance**
- but that table entry is an *estimate*: here it is $0$ while the true value of $\state_2$ is $1$
    - so the target $0$ points at the wrong place: **bias**
- this toy episode is deterministic, so nothing fluctuates in it
- the next slides measure both effects on the slippery lake


# The two targets side by side: bias and variance

**Why this slide?** We now have two ways to build a target. Ask both questions of each, on one state.

<img src="images/bias_variance_targets.png" width="900">

- start square of the slippery lake (each move goes as intended with probability $1/3$ and to either side with probability $1/3$ each), $\disc = 0.9$
- one fixed policy (the greedy policy of value iteration), 20,000 simulated samples of each target
- the TD target needs a table of estimates, and a table that is not yet right is what TD has *while it is learning*
    - for this picture we stand in for it with the table after 15 sweeps of last session's policy evaluation
    - in real TD the table is the current one, built by earlier updates
- the dashed line is the true value, $0.069$
    - the value of this fixed policy at the start square
    - computed from the model by solving the Bellman equation (as in the last session)


<table>
<thead><tr><th nowrap><div align="left"></div></th><th nowrap><div align="center">Monte Carlo target</div></th><th nowrap><div align="center">TD(0) target</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">average of the target</div></td><td nowrap><div align="center">$0.070$</div></td><td nowrap><div align="center">$0.042$</div></td></tr>
<tr><td nowrap><div align="left">bias (average minus true value $0.069$)</div></td><td nowrap><div align="center">about $0$</div></td><td nowrap><div align="center">$-0.027$</div></td></tr>
<tr><td nowrap><div align="left">spread (std)</div></td><td nowrap><div align="center">$0.113$</div></td><td nowrap><div align="center">$0.010$</div></td></tr>
</tbody></table>

- **Monte Carlo:** right on average, but a single target can be anywhere from $0$ to $0.59$
- **TD(0):** every target is close to every other, but they all point to the *wrong* place
    - the target is built from an estimate, and the estimate is not yet right: that is where its bias comes from
    - it fluctuates little because it depends on *one* random step, not on the whole episode
- **the TD bias is temporary**
    - it shrinks as the table improves: with the table after 40 sweeps the TD average is $0.068$
    - the variance of Monte Carlo does not shrink by itself
- **the trade-off:** TD pays a (shrinking) bias for a much smaller variance
    - MC pays no bias and a large variance


# How far ahead? A dial between TD and Monte Carlo

**Why this slide?** TD looks one step ahead, Monte Carlo looks to the end. Nothing forces us to choose one of the two extremes.

**$p$-step TD** uses
- the rewards of the next $p$ steps, $\{ \rewseq_{\tt'} \; | \;  \tt \lt \tt' \le \tt+p \}$
- plus the current estimate $\statevalfun (\stateseq_{\tt+p})$, in place of the rest of the return

$$
\delta = \left[ \sum_{i=1}^{p} \disc^{i-1} \rewseq_{\tt+i} + \disc^{p} \statevalfun(\stateseq_{\tt+p}) \right] - \statevalfun(\stateseq_{\tt})
$$

- $p = 1$ is TD(0)
- letting $p$ run to the end of the episode gives Monte Carlo


**Aside (for your own review): code for p-step TD**

The code below runs one episode.

The apparent complexity of the code arises
- because p-step TD, for $p \gt 1$
- involves returns that won't be experienced
- until $(p-1)$ steps after step $\tt$

So the update for $\statevalfun(\stateseq_\tt)$ is delayed $p-1$ steps
- until we accumulate the remaining rewards
- at the beginning of update $(k+1)$ we already have the value of $\statevalfun(\stateseq_{\tt+p})$ needed
    - i.e., $\statevalfun(\stateseq_{\tt+p})$ 

**Code**

- the code uses $p$ as the number of steps ahead, as on the slides
- it follows the Gymnasium interface (`reset()` returns `(state, info)`, `step()` returns five values)
- terminal states keep the value $0$; a time-out is treated like a terminal state

        from collections import defaultdict

        gamma = 0.9              # discount factor
        alpha = 0.5              # learning rate
        p = 3                    # number of steps ahead
        V = defaultdict(float)   # state-value function (terminal states stay at 0)

        def p_step_td_episode(env, policy, p):
            s, _ = env.reset()
            states, rewards = [s], []
            done = False

            while not done:
                a = policy(s)                          # policy(s) returns an action
                s, r, terminated, truncated, _ = env.step(a)
                done = terminated or truncated         # a time-out is treated like a terminal state (simplification)
                rewards.append(r)
                states.append(s)
                t = len(rewards)                       # steps taken so far

                if t >= p:                             # the state p steps back now has p rewards
                    u = t - p
                    G = sum(gamma**i * rewards[u + i] for i in range(p))
                    if not done:
                        G += gamma**p * V[states[t]]   # bootstrap from the current estimate
                    V[states[u]] += alpha * (G - V[states[u]])

            # end of episode: update the states that never got p rewards (terminal value is 0)
            T = len(rewards)
            for u in range(max(0, T - p + 1), T):
                G = sum(gamma**i * rewards[u + i] for i in range(T - u))
                V[states[u]] += alpha * (G - V[states[u]])


<table>
<thead><tr><th nowrap><div align="left">Method</div></th><th nowrap><div align="left">Update basis</div></th><th nowrap><div align="left">Target</div></th><th nowrap><div align="left">Bias</div></th><th nowrap><div align="left">Variance</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">TD 1-step</div></td><td nowrap><div align="left">after 1 step: immediate reward + next value</div></td><td nowrap><div align="left">$\rewseq_{\tt+1} + \disc\,\statevalfun(\stateseq_{\tt+1})$</div></td><td nowrap><div align="left">most (all the rest of the return is an estimate)</div></td><td nowrap><div align="left">least (one random step)</div></td></tr>
<tr><td nowrap><div align="left">TD p-step</div></td><td nowrap><div align="left">after p steps: p rewards + value at $\tt+p$</div></td><td nowrap><div align="left">$\rewseq_{\tt+1} + \ldots + \disc^{p-1} \rewseq_{\tt+p} + \disc^{p} \statevalfun(\stateseq_{\tt+p})$</div></td><td nowrap><div align="left">smaller as $p$ grows (more of the target is real rewards, less is estimate)</div></td><td nowrap><div align="left">grows with $p$ (more random steps)</div></td></tr>
<tr><td nowrap><div align="left">MC ($p \to \infty$)</div></td><td nowrap><div align="left">end of episode: full return</div></td><td nowrap><div align="left">$\rlret_\tt$</div></td><td nowrap><div align="left">none</div></td><td nowrap><div align="left">most</div></td></tr>
</tbody></table>

- **$p$ is a dial, not a switch**: small $p$ buys low variance with some bias
    - large $p$ buys low bias with high variance (the reasoning above, not something we measure here)
- the same dial returns when we learn the policy directly (advantage estimates)


# Interlude: the shape of a backup

**Why this interlude?** We now have DP, TD, $p$-step TD and Monte Carlo.

Recall the definition (A1): a **backup** is an update of a value from the values of what follows it. This picture puts the methods of the last two sessions on one grid.

> **Two choices describe every backup: how many successors (width), and how far ahead (depth).**

<img src="images/backup_shape.png" width="820">

- **how to read the pictures:** each circle is a state
    - the black one is the state being updated
    - the white ones are the states its update looks at
        - for dynamic programming the actions and their outcomes are both averaged, drawn as one level

**Width and depth**

- **width:** use *all* successors weighted by their probabilities (needs the model), or *one sampled* successor
- **depth:** look one step ahead (then use an estimate), $p$ steps ahead, or to the end of the episode (no estimate)
- last session was the top row (dynamic programming, depth one)
- this session is the bottom row
- all successors, to the end, would need the model and is too expensive for any problem of interest


## Converging to the true $\statevalfun(\state)$

**Why this slide?** Samples are random, so a single target can be far off. Do the learned values still end up right? Under what conditions?

> **Averaging many sampled targets converges on their expectation, which is the value once the estimates inside the target are right. Every state must be visited again and again.**

- we never observe $\transpm(\state', \rew \mid \state, \act)$: each step gives one random sample $(\rewseq_{\tt+1}, \stateseq_{\tt+1})$ drawn from it
- the samples must cover what $\transpm$ can produce, in the right proportions
$$
\Exp{\pi}\left[ \rewseq_{\tt+1} + \disc \, \statevalfun_\pi(\stateseq_{\tt+1}) \mid \stateseq_\tt = \state \right] = \statevalfun_\pi(\state)
$$
- careful: this holds with the *true* $\statevalfun_\pi$ inside
    - a TD target contains the *current* estimate, so its average is off until the estimates are right (the bias of the earlier slide)
- a constant $\alpha$ keeps the estimate bouncing around the value, because every target fluctuates
    - a decaying learning rate averages the fluctuation away (our examples use a constant $\alpha$)


**Two different reasons to iterate**

> **Dependence (a value rests on values not yet right) and sampling (each update is one random sample).**

- *dependence*: a value depends on values we do not know yet (present even with a model)
- *sampling*: without a model each update is one random sample
    - the agent must explore to reach the reward at all
    - this reason applies only without a model
- with a model we can sit in our chair and compute
- without one, the agent must go and find out


# What we established, and what it sets up

- **Established**
    - without a model, values are learned from sampled episodes
    - a target is a random number
    - **bias** (does its average point at the true value?) and **variance** (how much does it fluctuate?) say how good it is
    - Monte Carlo: unbiased, high variance, waits for the episode to end
    - TD(0): *bootstrapped* target, biased while the estimates are wrong, low variance, updates after every step
    - $p$-step TD is a dial between the two
    - width (all or one successor) and depth (how far ahead) describe every backup
    - **model ledger:** $\transpm$ is gone from the value updates, because samples replace the sum over successors
    - the book's own algorithm boxes, in its notation: first-visit Monte Carlo prediction (Sutton and Barto section 5.1) and tabular TD(0) (section 6.1)
- **Sets up**
    - next session: we have only *evaluated* a policy. To *act*, we need action values, exploration, and Q-learning (the sampled value iteration)
    - **the hand-off:** the model has left value estimation, but acting greedily on $\statevalfun$ still needs it
        - acting needs a lookahead through $\transpm$
        - that is why A4 moves to $\actvalfun$
    - the question we kept in mind: the episodes came from the policy being evaluated. Must they? (on-policy versus off-policy, with SARSA and Q-learning)
    - later: bias and variance return when we learn the policy directly (baseline, advantage)
